# XGBoost Multiclass Sequence Locator Training (Day 3)

This notebook covers the training and evaluation of the XGBoost classifier to locate 8 key golf swing milestones from sliding window features. We address class imbalance using inverse-frequency sample weights, split the data by video ID to prevent leakage, and verify the model visually using OpenCV.

In [ ]:
# Import required libraries
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import cv2

# Set project root path relative to this notebook's directory
PROJECT_ROOT = os.path.abspath("../")
sys.path.append(PROJECT_ROOT)

## 1. Load Master Dataset

We load the engineered master training dataset that contains the raw and normalized coordinates along with the sliding window features ($\pm 5$ frame offsets) and metadata labels.

In [ ]:
dataset_path = os.path.join(PROJECT_ROOT, "data/processed/master_dataset.csv")
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Master dataset not found at {dataset_path}")

print("Loading master dataset...")
df_master = pd.read_csv(dataset_path)
print(f"Loaded master dataset with shape: {df_master.shape}")

# Filter for golf videos only
df_master = df_master[df_master["is_golf"] == 1].copy()
print(f"Filtered for golf videos only: {df_master.shape}")

## 2. Perform Group Train/Val/Test Split on Video ID (80/10/10)

To prevent data leakage, we perform a group split on unique `video_id`s, ensuring that frames from the same video are kept entirely in either the train, validation, or test sets. Shuffling is done using a fixed random seed (`42`) for reproducibility.

In [ ]:
# Randomly split on video_id using a fixed seed (42)
unique_video_ids = df_master["video_id"].unique()
print(f"Total unique videos: {len(unique_video_ids)}")

np.random.seed(42)
np.random.shuffle(unique_video_ids)

num_videos = len(unique_video_ids)
split_train = int(num_videos * 0.8)
split_val = int(num_videos * 0.9)

train_vids = unique_video_ids[:split_train]
val_vids = unique_video_ids[split_train:split_val]
test_vids = unique_video_ids[split_val:]

print(f"Train videos: {len(train_vids)} ({len(train_vids)/num_videos:.1%})")
print(f"Val videos: {len(val_vids)} ({len(val_vids)/num_videos:.1%})")
print(f"Test videos: {len(test_vids)} ({len(test_vids)/num_videos:.1%})")

# Filter dataframes based on split video IDs
df_train = df_master[df_master["video_id"].isin(train_vids)].copy()
df_val = df_master[df_master["video_id"].isin(val_vids)].copy()
df_test = df_master[df_master["video_id"].isin(test_vids)].copy()

print(f"\nTrain frames: {len(df_train)}")
print(f"Val frames: {len(df_val)}")
print(f"Test frames: {len(df_test)}")

## 3. Print and Compare Distribution Statistics

We verify that the label distributions (classes 0-8) and categorical metadata fields (views, genders, club types) remain balanced and representative across the original dataset and all three split sets.

In [ ]:
def compare_distributions(dfs, names):
    stats = []
    
    for df, name in zip(dfs, names):
        total_frames = len(df)
        total_vids = df["video_id"].nunique()
        
        row = {"Split": name, "Videos": total_vids, "Frames": total_frames}
        
        # Label breakdown
        label_counts = df["label"].value_counts(normalize=True).sort_index().to_dict()
        for lbl in range(9):
            row[f"Class_{lbl}_%"] = f"{label_counts.get(lbl, 0.0) * 100:.2f}%"
            
        # Metadata breakdown
        for col in ["view", "sex", "club"]:
            if col in df.columns:
                val_counts = df[col].value_counts(normalize=True).to_dict()
                for val, ratio in val_counts.items():
                    row[f"{col}_{val}_%"] = f"{ratio * 100:.1f}%"
                    
        stats.append(row)
        
    df_stats = pd.DataFrame(stats).T
    return df_stats

dfs = [df_master, df_train, df_val, df_test]
names = ["Original", "Train", "Val", "Test"]
df_comparison = compare_distributions(dfs, names)
df_comparison

## 4. Save Split Datasets (dropping raw_ and smooth_ columns to save space)

We drop the unused 3D coordinate pixel positions (`raw_*` and `smooth_*`) to save memory and optimize file sizes. Then we export each split to a separate CSV under `data/processed/`.

In [ ]:
cols_to_keep = [c for c in df_master.columns if not (c.startswith("raw_") or c.startswith("smooth_"))]

print(f"Original columns count: {df_master.shape[1]}")
print(f"Filtered columns count: {len(cols_to_keep)}")

df_train_save = df_train[cols_to_keep]
df_val_save = df_val[cols_to_keep]
df_test_save = df_test[cols_to_keep]

out_dir = os.path.join(PROJECT_ROOT, "data/processed")
os.makedirs(out_dir, exist_ok=True)

train_path = os.path.join(out_dir, "train_dataset.csv")
val_path = os.path.join(out_dir, "val_dataset.csv")
test_path = os.path.join(out_dir, "test_dataset.csv")

print("Saving split datasets...")
df_train_save.to_csv(train_path, index=False)
df_val_save.to_csv(val_path, index=False)
df_test_save.to_csv(test_path, index=False)

print(f"Saved Train: {train_path} ({os.path.getsize(train_path)/1024/1024:.2f} MB)")
print(f"Saved Val: {val_path} ({os.path.getsize(val_path)/1024/1024:.2f} MB)")
print(f"Saved Test: {test_path} ({os.path.getsize(test_path)/1024/1024:.2f} MB)")

## 4.5. Reload Split Datasets (Fast Entrypoint)

If you have already run the splitting and saving cells above once, you can bypass them on subsequent runs and start directly here. This loads the saved, memory-optimized train, validation, and test datasets from disk.

In [ ]:
# Load pre-split datasets directly to skip the heavy loading/splitting steps above
train_path = os.path.join(PROJECT_ROOT, "data/processed/train_dataset.csv")
val_path = os.path.join(PROJECT_ROOT, "data/processed/val_dataset.csv")
test_path = os.path.join(PROJECT_ROOT, "data/processed/test_dataset.csv")

print("Loading pre-split datasets from disk...")
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

print(f"Loaded Train set: {df_train.shape}")
print(f"Loaded Val set:   {df_val.shape}")
print(f"Loaded Test set:  {df_test.shape}")

## 5. Feature Selection and Dtype Casting

We select the 98 normalized coordinates and offsets along with 3 categorical fields as our training features. Categorical features are cast to pandas `category` dtype for native XGBoost support.

In [ ]:
# Find all normalized coordinate columns in the reloaded dataframes
feature_cols = sorted([c for c in df_train.columns if c.startswith("norm_")])
categorical_cols = ["view", "sex", "club"]
all_feature_cols = feature_cols + categorical_cols

print(f"Pose features (norm_): {len(feature_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Total features: {len(all_feature_cols)}")

# Cast categoricals to category dtype
for col in categorical_cols:
    df_train[col] = df_train[col].astype("category")
    df_val[col] = df_val[col].astype("category")
    df_test[col] = df_test[col].astype("category")

X_train = df_train[all_feature_cols]
y_train = df_train["label"]

X_val = df_val[all_feature_cols]
y_val = df_val["label"]

X_test = df_test[all_feature_cols]
y_test = df_test["label"]

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

## 6. Calculate Sample Weights to Balance Class Imbalance

Since transitional class 0 represents ~97% of all frames, we compute inverse frequency weights to force the model to penalize errors on rare milestone frames (classes 1-8).

In [ ]:
class_counts = y_train.value_counts().sort_index()
total_samples = len(y_train)
n_classes = len(class_counts)

# Inverse-frequency weight formula: total_samples / (n_classes * class_count)
class_weights = {}
for cls, count in class_counts.items():
    class_weights[cls] = total_samples / (n_classes * count)

print("Calculated inverse-frequency class weights:")
for cls, w in sorted(class_weights.items()):
    print(f"Class {cls}: {w:.4f} (Count: {class_counts[cls]})")

# Map class weights to training samples
sample_weights = y_train.map(class_weights)
print(f"Sample weights mapped to {len(sample_weights)} rows.")

## 7. Train XGBoost Multiclass Classifier

We train an `XGBClassifier` with the `multi:softprob` objective, enabling native categorical support and early stopping based on validation loss.

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=9,
    tree_method="hist",
    enable_categorical=True,
    random_state=42
)

print("Training XGBClassifier with validation set early stopping...")
model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=10
)
print("Training completed!")

## 8. Save the Model

We serialize the trained booster to JSON format using the native XGBoost API.

In [ ]:
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, "golf_phase_model.json")

model.save_model(model_path)
print(f"Saved serialized model to: {model_path}")

## 9. Evaluate Classification Metrics on Test Set

We compute the final classification report (precision, recall, and F1-score) on the holdout test set to ensure high quality on unseen data.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)
print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")

event_names = {
    0: 'Transition', 1: 'Address', 2: 'Toe-up (Backswing)', 3: 'Mid-backswing',
    4: 'Top of Backswing', 5: 'Mid-downswing', 6: 'Impact', 7: 'Mid-follow-through', 8: 'Finish'
}
target_names = [event_names[i] for i in range(9)]

print("Test Set Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

## 10. Implement and Test predict_swing_milestones()

We define our core inference function `predict_swing_milestones()` and test it by comparing predictions against ground-truth labels on a test set video. The prediction takes argmax over the predicted probabilities of classes 1 to 8 across all frames to find the peak frame index for each event.

In [ ]:
# Import the verified inference function from src/train_classifier.py to avoid code duplication
from src.train_classifier import predict_swing_milestones

In [ ]:
# Test on a test set video
test_video_ids = df_test["video_id"].unique()
test_vid = test_video_ids[0]
print(f"Testing inference on video_id: {test_vid}")

df_vid_features = df_test[df_test["video_id"] == test_vid].sort_values("frame_index").copy()

# Recover true milestones from the label column
true_milestones = []
for c in range(1, 9):
    row = df_vid_features[df_vid_features["label"] == c]
    if not row.empty:
        true_milestones.append(int(row.iloc[0]["frame_index"]))
    else:
        true_milestones.append(-1)
        
predicted_milestones = predict_swing_milestones(df_vid_features, model)

print(f"True milestones:      {true_milestones}")
print(f"Predicted milestones: {predicted_milestones}")
errors = [abs(p - t) for p, t in zip(predicted_milestones, true_milestones)]
print(f"Absolute errors:      {errors}")
print(f"Mean Absolute Error:  {np.mean(errors):.2f} frames")

## 10.5. Evaluate Peak-Finding Errors across the entire Test Set

To get a global view of model performance, we run the peak-finding inference across all videos in the test set. We calculate the Mean Absolute Error (MAE) and maximum absolute error for each milestone, along with identifying the worst-performing video for each event.

In [ ]:
def evaluate_test_set_peaks(df_test_data, trained_model):
    test_video_ids = df_test_data["video_id"].unique()
    print(f"Evaluating {len(test_video_ids)} test videos...")
    
    all_errors = []
    event_names = {
        1: 'Address', 2: 'Toe-up (Backswing)', 3: 'Mid-backswing',
        4: 'Top of Backswing', 5: 'Mid-downswing', 6: 'Impact',
        7: 'Mid-follow-through', 8: 'Finish'
    }
    
    for vid in test_video_ids:
        # Get features for this video, sorted by frame_index
        df_vid = df_test_data[df_test_data["video_id"] == vid].sort_values("frame_index").copy()
        
        # Recover true milestone frame indices
        true_indices = []
        for c in range(1, 9):
            row = df_vid[df_vid["label"] == c]
            if not row.empty:
                true_indices.append(int(row.iloc[0]["frame_index"]))
            else:
                true_indices.append(-1)
                
        if -1 in true_indices:
            # Skip if any milestone is missing in labels
            continue
            
        # Predict milestone frame indices using argmax peak-finding
        pred_indices = predict_swing_milestones(df_vid, trained_model)
        
        # Compute frame errors (absolute and signed)
        abs_errors = [abs(p - t) for p, t in zip(pred_indices, true_indices)]
        signed_errors = [p - t for p, t in zip(pred_indices, true_indices)]
        
        all_errors.append({
            "video_id": vid,
            "abs_errors": abs_errors,
            "signed_errors": signed_errors,
            "mean_error": np.mean(abs_errors)
        })
        
    df_errors = pd.DataFrame(all_errors)
    
    # Identify the worst overall video by average MAE
    worst_overall_idx = df_errors["mean_error"].idxmax()
    worst_overall_vid = df_errors.loc[worst_overall_idx, "video_id"]
    worst_overall_mae = df_errors.loc[worst_overall_idx, "mean_error"]
    print(f"Worst Overall Video: ID {int(worst_overall_vid)} with a Mean Absolute Error of {worst_overall_mae:.2f} frames.\n")
    
    # Build the milestone summary table
    summary_rows = []
    for c in range(1, 9):
        # Extract errors for this milestone across all test videos
        m_abs_errors = [x["abs_errors"][c-1] for x in all_errors]
        m_signed_errors = [x["signed_errors"][c-1] for x in all_errors]
        
        mae = np.mean(m_abs_errors)
        med_ae = np.median(m_abs_errors)
        mean_bias = np.mean(m_signed_errors)
        max_err = np.max(m_abs_errors)
        
        # Find the video ID that caused the max error for this milestone
        max_err_idx = np.argmax(m_abs_errors)
        worst_vid_for_milestone = all_errors[max_err_idx]["video_id"]
        
        summary_rows.append({
            "Milestone": f"{c}: {event_names[c]}",
            "MAE (frames)": f"{mae:.2f}",
            "Median AE (frames)": f"{med_ae:.1f}",
            "Mean Bias (frames)": f"{mean_bias:+.2f}",
            "Max Error (frames)": int(max_err),
            "Worst Video ID": int(worst_vid_for_milestone)
        })
        
    df_summary = pd.DataFrame(summary_rows)
    return df_summary

df_summary_stats = evaluate_test_set_peaks(df_test, model)
df_summary_stats

## 11. Generate Verification Video

We generate a visual verification overlay video by flashing green predictions and red true milestones on the frames of a test swing using OpenCV. This allows us to manually verify that predicted events align with the physical swing progression.

In [ ]:
def generate_verification_video(video_id, model, df_split, out_path):
    video_path = os.path.join(PROJECT_ROOT, f"data/videos_160/videos_160/{video_id}.mp4")
    if not os.path.exists(video_path):
        video_path = os.path.join(PROJECT_ROOT, f"data/videos_160/{video_id}.mp4")
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video file not found for video_id {video_id}")
            
    df_vid = df_split[df_split["video_id"] == video_id].sort_values("frame_index").copy()
    pred_indices = predict_swing_milestones(df_vid, model)
    
    true_indices = []
    for c in range(1, 9):
        row = df_vid[df_vid["label"] == c]
        if not row.empty:
            true_indices.append(int(row.iloc[0]["frame_index"]))
        else:
            true_indices.append(-1)
            
    event_names = {
        1: 'ADDRESS', 2: 'TOE-UP (BACK)', 3: 'MID-BACK',
        4: 'TOP', 5: 'MID-DOWN', 6: 'IMPACT', 7: 'RELEASE', 8: 'FINISH'
    }
    
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Overlay predictions (green, top)
        for c_idx, p_frame in enumerate(pred_indices):
            class_label = c_idx + 1
            if abs(frame_idx - p_frame) <= 1:
                name = event_names[class_label]
                cv2.putText(frame, f"PRED: {name}", (10, 40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)
                
        # Overlay true labels (red, bottom)
        for c_idx, t_frame in enumerate(true_indices):
            class_label = c_idx + 1
            if frame_idx == t_frame:
                name = event_names[class_label]
                cv2.putText(frame, f"TRUE: {name}", (10, height - 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)
                
        out.write(frame)
        frame_idx += 1
        
    cap.release()
    out.release()
    print(f"Verification video successfully generated at: {out_path}")

demo_out_path = os.path.join(PROJECT_ROOT, "data/processed/milestone_flash_demo.mp4")
generate_verification_video(test_vid, model, df_test, demo_out_path)